#### We have 2 types of Categorical Data
* Nominal Data (without any ranking)
* Ordinal Data (defined ranking)

In [1]:
import pandas as pd

df = pd.DataFrame({
    "age": [21, 25, 30, 22, 28, 35, 24, 31],
    "city": [
        "Baku", "Milan", "London", "Baku",
        "Paris", "London", "Milan", "Paris"
    ],
    "gender": [
        "Male", "Female", "Female", "Male",
        "Male", "Female", "Male", "Female"
    ],
    "education": [
        "Bachelor", "Master", "PhD", "Bachelor",
        "Master", "High School", "Bachelor", "PhD"
    ],
    "experience_level": [
        "Junior", "Mid", "Senior", "Junior",
        "Mid", "Junior", "Senior", "Mid"
    ],
    "remote": [
        "No", "Yes", "Yes", "No",
        "Yes", "No", "Yes", "Yes"
    ],
    "salary": [
        900, 1500, 3000, 1000,
        2200, 1200, 2800, 2000
    ],
    "job": [
        "Data Analyst", "Data Scientist", "AI Engineer", "Data Analyst",
        "Data Analyst", "AI Engineer", "Data Scientist", "Data Scientist"
    ]
})

df

,age,city,gender,education,experience_level,remote,salary,job
0,21,Baku,Male,Bachelor,Junior,No,900,Data Analyst
1,25,Milan,Female,Master,Mid,Yes,1500,Data Scientist
2,30,London,Female,PhD,Senior,Yes,3000,AI Engineer
3,22,Baku,Male,Bachelor,Junior,No,1000,Data Analyst
4,28,Paris,Male,Master,Mid,Yes,2200,Data Analyst
5,35,London,Female,High School,Junior,No,1200,AI Engineer
6,24,Milan,Male,Bachelor,Senior,Yes,2800,Data Scientist
7,31,Paris,Female,PhD,Mid,Yes,2000,Data Scientist


In [2]:
df_encoded = df.copy()

### Label Encoding

* Used in tree-based models like Decision Trees or XGBoost.
* Pros: Simple and memory-efficient.
* Cons: Introduces implicit order which may be misinterpreted by non-tree models when used with nominal data.

In [3]:
from sklearn.preprocessing import LabelEncoder

df_encoded['remote'] = LabelEncoder().fit_transform(df_encoded['remote'])

In [4]:
df_encoded

,age,city,gender,education,experience_level,remote,salary,job
0,21,Baku,Male,Bachelor,Junior,0,900,Data Analyst
1,25,Milan,Female,Master,Mid,1,1500,Data Scientist
2,30,London,Female,PhD,Senior,1,3000,AI Engineer
3,22,Baku,Male,Bachelor,Junior,0,1000,Data Analyst
4,28,Paris,Male,Master,Mid,1,2200,Data Analyst
5,35,London,Female,High School,Junior,0,1200,AI Engineer
6,24,Milan,Male,Bachelor,Senior,1,2800,Data Scientist
7,31,Paris,Female,PhD,Mid,1,2000,Data Scientist


### One-Hot Encoding

* Used in linear models, logistic regression and neural networks.
* Pros: Does not assume order; widely supported.
* Cons: Can cause high dimensionality and sparse data when feature has many categories.

In [5]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

In [6]:
one_hot = encoder.fit_transform(df_encoded[['city']])

In [7]:
one_hot

array([[1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

In [8]:
encoded = pd.DataFrame(
    one_hot, 
    columns=encoder.get_feature_names_out(['city']),
    index=df_encoded.index
)

In [9]:
encoded

,city_Baku,city_London,city_Milan,city_Paris
0,1.0,0.0,0.0,0.0
1,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0
3,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0
5,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0
7,0.0,0.0,0.0,1.0


In [10]:
city_position = df.columns.get_loc('city')
city_position

1

In [11]:
df_encoded = df_encoded.drop(columns=['city'])

In [12]:
for i, column in enumerate(encoded.columns):
    df_encoded.insert(city_position + i, column, encoded[column])

##### In previous stages I just wanted to keep city position static and new columns that we replaced with city also went to same position that city was in

In [13]:
df_encoded

,age,city_Baku,city_London,city_Milan,city_Paris,gender,education,experience_level,remote,salary,job
0,21,1.0,0.0,0.0,0.0,Male,Bachelor,Junior,0,900,Data Analyst
1,25,0.0,0.0,1.0,0.0,Female,Master,Mid,1,1500,Data Scientist
2,30,0.0,1.0,0.0,0.0,Female,PhD,Senior,1,3000,AI Engineer
3,22,1.0,0.0,0.0,0.0,Male,Bachelor,Junior,0,1000,Data Analyst
4,28,0.0,0.0,0.0,1.0,Male,Master,Mid,1,2200,Data Analyst
5,35,0.0,1.0,0.0,0.0,Female,High School,Junior,0,1200,AI Engineer
6,24,0.0,0.0,1.0,0.0,Male,Bachelor,Senior,1,2800,Data Scientist
7,31,0.0,0.0,0.0,1.0,Female,PhD,Mid,1,2000,Data Scientist


### We can also use get_dummies for same purpose which is way more easier

In [14]:
one_hot_with_dummies = pd.get_dummies(df['city'])

In [15]:
one_hot_with_dummies

,Baku,London,Milan,Paris
0,True,False,False,False
1,False,False,True,False
2,False,True,False,False
3,True,False,False,False
4,False,False,False,True
5,False,True,False,False
6,False,False,True,False
7,False,False,False,True


### Ordinal Encoding

* Used for ordered features like ratings or education levels.
* Pros: Maintains order; reduces dimensionality.
* Cons: Not suitable for nominal categories.

In [16]:
from sklearn.preprocessing import OrdinalEncoder

In [17]:
df_encoded['experience_level'] = OrdinalEncoder(categories=[['Junior', 'Mid', 'Senior']]).fit_transform(df_encoded[['experience_level']])

In [18]:
df_encoded

,age,city_Baku,city_London,city_Milan,city_Paris,gender,education,experience_level,remote,salary,job
0,21,1.0,0.0,0.0,0.0,Male,Bachelor,0.0,0,900,Data Analyst
1,25,0.0,0.0,1.0,0.0,Female,Master,1.0,1,1500,Data Scientist
2,30,0.0,1.0,0.0,0.0,Female,PhD,2.0,1,3000,AI Engineer
3,22,1.0,0.0,0.0,0.0,Male,Bachelor,0.0,0,1000,Data Analyst
4,28,0.0,0.0,0.0,1.0,Male,Master,1.0,1,2200,Data Analyst
5,35,0.0,1.0,0.0,0.0,Female,High School,0.0,0,1200,AI Engineer
6,24,0.0,0.0,1.0,0.0,Male,Bachelor,2.0,1,2800,Data Scientist
7,31,0.0,0.0,0.0,1.0,Female,PhD,1.0,1,2000,Data Scientist


### Target Encoding

* Useful for high-cardinality categorical features when categories occur repeatedly and have meaningful relationships with the target (e.g., ZIP codes or frequently occurring product IDs).
* Pros: Can represent target-related information compactly, especially when one-hot encoding would create thousands of columns.
* Cons: High risk of target leakage and overfitting; requires careful train/test handling, often with out-of-fold encoding and smoothing.

In [19]:
from sklearn.preprocessing import TargetEncoder

In [20]:
df_encoded['education'] = TargetEncoder(target_type='continuous')\
.fit_transform(df_encoded[['education']], df_encoded['salary'])

In [21]:
df_encoded

,age,city_Baku,city_London,city_Milan,city_Paris,gender,education,experience_level,remote,salary,job
0,21,1.0,0.0,0.0,0.0,Male,1956.413233,0.0,0,900,Data Analyst
1,25,0.0,0.0,1.0,0.0,Female,2200.000000,1.0,1,1500,Data Scientist
2,30,0.0,1.0,0.0,0.0,Female,2000.000000,2.0,1,3000,AI Engineer
3,22,1.0,0.0,0.0,0.0,Male,1950.780243,0.0,0,1000,Data Analyst
4,28,0.0,0.0,0.0,1.0,Male,1500.000000,1.0,1,2200,Data Analyst
5,35,0.0,1.0,0.0,0.0,Female,2066.666667,0.0,0,1200,AI Engineer
6,24,0.0,0.0,1.0,0.0,Male,952.689416,2.0,1,2800,Data Scientist
7,31,0.0,0.0,0.0,1.0,Female,3000.000000,1.0,1,2000,Data Scientist


### Binary Encoding

* Applied in high-cardinality text/NLP tasks to save memory.
* Pros: Reduces dimensionality, more memory-efficient than one-hot encoding.
* Cons: Slightly more complex; requires careful handling of missing values.

In [22]:
from category_encoders import BinaryEncoder

In [23]:
df_encoded = BinaryEncoder(cols=['gender']).fit_transform(df_encoded)

In [24]:
df_encoded

,age,city_Baku,city_London,city_Milan,city_Paris,gender_0,gender_1,education,experience_level,remote,salary,job
0,21,1.0,0.0,0.0,0.0,0,1,1956.413233,0.0,0,900,Data Analyst
1,25,0.0,0.0,1.0,0.0,1,0,2200.000000,1.0,1,1500,Data Scientist
2,30,0.0,1.0,0.0,0.0,1,0,2000.000000,2.0,1,3000,AI Engineer
3,22,1.0,0.0,0.0,0.0,0,1,1950.780243,0.0,0,1000,Data Analyst
4,28,0.0,0.0,0.0,1.0,0,1,1500.000000,1.0,1,2200,Data Analyst
5,35,0.0,1.0,0.0,0.0,1,0,2066.666667,0.0,0,1200,AI Engineer
6,24,0.0,0.0,1.0,0.0,0,1,952.689416,2.0,1,2800,Data Scientist
7,31,0.0,0.0,0.0,1.0,1,0,3000.000000,1.0,1,2000,Data Scientist


### Frequency Encoding

* Effective in retail, e-commerce or clickstream data for popularity trends.
* Pros: Low computational and storage requirements.
* Cons: Can introduce data leakage if not handled properly.

In [25]:
from category_encoders import CountEncoder

In [26]:
df_encoded['job'] = CountEncoder().fit_transform(df_encoded['job'])

In [27]:
df

,age,city,gender,education,experience_level,remote,salary,job
0,21,Baku,Male,Bachelor,Junior,No,900,Data Analyst
1,25,Milan,Female,Master,Mid,Yes,1500,Data Scientist
2,30,London,Female,PhD,Senior,Yes,3000,AI Engineer
3,22,Baku,Male,Bachelor,Junior,No,1000,Data Analyst
4,28,Paris,Male,Master,Mid,Yes,2200,Data Analyst
5,35,London,Female,High School,Junior,No,1200,AI Engineer
6,24,Milan,Male,Bachelor,Senior,Yes,2800,Data Scientist
7,31,Paris,Female,PhD,Mid,Yes,2000,Data Scientist


In [28]:
df_encoded

,age,city_Baku,city_London,city_Milan,city_Paris,gender_0,gender_1,education,experience_level,remote,salary,job
0,21,1.0,0.0,0.0,0.0,0,1,1956.413233,0.0,0,900,3
1,25,0.0,0.0,1.0,0.0,1,0,2200.000000,1.0,1,1500,3
2,30,0.0,1.0,0.0,0.0,1,0,2000.000000,2.0,1,3000,2
3,22,1.0,0.0,0.0,0.0,0,1,1950.780243,0.0,0,1000,3
4,28,0.0,0.0,0.0,1.0,0,1,1500.000000,1.0,1,2200,3
5,35,0.0,1.0,0.0,0.0,1,0,2066.666667,0.0,0,1200,2
6,24,0.0,0.0,1.0,0.0,0,1,952.689416,2.0,1,2800,3
7,31,0.0,0.0,0.0,1.0,1,0,3000.000000,1.0,1,2000,3


In [30]:
import pandas as pd

target_corr = df_encoded.corr(method="pearson")["salary"]

target_corr = target_corr.drop("salary")

corr_df = pd.DataFrame(
    {
        "Correlation (Direction)": target_corr,
        "Impact Strength (Absolute)": target_corr.abs(),
    }
)


sorted_features = corr_df.sort_values(
    by="Impact Strength (Absolute)", ascending=False
)

print(sorted_features)

                  Correlation (Direction)  Impact Strength (Absolute)
experience_level                 0.962371                    0.962371
remote                           0.814469                    0.814469
city_Baku                       -0.670971                    0.670971
education                       -0.322579                    0.322579
age                              0.256677                    0.256677
city_Milan                       0.249218                    0.249218
city_Paris                       0.210877                    0.210877
city_London                      0.210877                    0.210877
job                             -0.210877                    0.210877
gender_1                        -0.132818                    0.132818
gender_0                         0.132818                    0.132818
